In [ ]:
!pip install unsloth -q
!pip install trl peft accelerate bitsandbytes datasets kaggle pyarrow -q

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Secrets (key icon) -> add HF_TOKEN
HF_TOKEN = userdata.get("HF_TOKEN")
HF_USERNAME = "Bharat773"

login(HF_TOKEN)
print("successful")


In [ ]:
from google.colab import files
import os, json

print("kaggle.json")
uploaded = files.upload()

os.makedirs("/root/.kaggle", exist_ok=True)

for filename, content in uploaded.items():
    with open("/root/.kaggle/kaggle.json", "wb") as f:
        f.write(content)

os.chmod("/root/.kaggle/kaggle.json", 0o600)


with open("/root/.kaggle/kaggle.json") as f:
    creds = json.load(f)
    print(f"Logged in as: {creds.get('username', 'unknown')}")

In [ ]:
import os

BASE = "/content/nutrition5k"
os.makedirs(BASE, exist_ok=True)

FILES = [
    "dish_ingredients.csv",
    "dish_nutrition_values.csv",
    "ingredients_metadata.csv",
]

print("Downloading")
for f in FILES:
    os.system(f"kaggle datasets download -d gillesokhin/nutrition5k-dataset --file {f} -p {BASE} -q")
    path = f"{BASE}/{f}"
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print(f"    {f} - {size_kb:.1f} KB")
    else:
        print(f"    {f} - failed")

In [ ]:
import pandas as pd

BASE = "/content/nutrition5k"

df_ing  = pd.read_csv(f"{BASE}/dish_ingredients.csv")
df_nut  = pd.read_csv(f"{BASE}/dish_nutrition_values.csv")
df_meta = pd.read_csv(f"{BASE}/ingredients_metadata.csv")

print(" dish_ingredients.csv columns:", list(df_ing.columns))
print(" dish_nutrition_values.csv columns:", list(df_nut.columns))
print()
print(f"   Ingredient rows : {len(df_ing):,}")
print(f"   Dishes          : {df_nut.shape[0]:,}")
print(f"   Unique ingredients: {df_meta.shape[0]:,}")
print()


print("Sample from dish_ingredients.csv:")
print(df_ing.head(4).to_string(index=False))
print()
print("Sample from dish_nutrition_values.csv:")
print(df_nut.head(3).to_string(index=False))

In [ ]:
import json, re, pandas as pd

BASE = "/content/nutrition5k"


df_ing = pd.read_csv(f"{BASE}/dish_ingredients.csv")
df_nut = pd.read_csv(f"{BASE}/dish_nutrition_values.csv")

def find_col(df, *candidates):
    for c in candidates:
        for col in df.columns:
            if col.lower() == c.lower():
                return col
    raise KeyError(f"None of {candidates} found in columns: {list(df.columns)}")

ING_ID    = find_col(df_ing, "dish_id", "id")
ING_NAME  = find_col(df_ing, "ingredient_name", "ingr_name", "name")
ING_GRAMS = find_col(df_ing, "grams", "ingr_grams", "weight_g", "weight")

NUT_ID    = find_col(df_nut, "dish_id", "id")
NUT_CAL   = find_col(df_nut, "total_calories", "calories")
NUT_MASS  = find_col(df_nut, "total_mass", "mass", "total_weight")
NUT_FAT   = find_col(df_nut, "total_fat", "fat")
NUT_CARB  = find_col(df_nut, "total_carb", "total_carbs", "carbs", "carbohydrates", "carb")
NUT_PRO   = find_col(df_nut, "total_protein", "protein")

print(f" Columns mapped:")
print(f"   ingredients → dish_id={ING_ID}, name={ING_NAME}, grams={ING_GRAMS}")
print(f"   nutrition   → dish_id={NUT_ID}, cal={NUT_CAL}, mass={NUT_MASS}")


grouped = (
    df_ing
    .groupby(ING_ID)
    .apply(lambda g: {
        "ingredients": g[ING_NAME].tolist(),
        "grams":       g[ING_GRAMS].tolist(),
    })
    .reset_index(name="info")
)
grouped["ingredients"] = grouped["info"].apply(lambda x: x["ingredients"])
grouped["grams_list"]  = grouped["info"].apply(lambda x: x["grams"])
grouped = grouped.drop(columns="info").rename(columns={ING_ID: "dish_id"})

df = grouped.merge(
    df_nut[[NUT_ID, NUT_CAL, NUT_MASS, NUT_FAT, NUT_CARB, NUT_PRO]],
    left_on="dish_id", right_on=NUT_ID,
    how="inner"
)
print(f"\n {len(df):,} dishes with ingredients + nutrition")


def make_dish_name(ingredients, grams_list):
    """Name a dish from its top 3 ingredients by weight."""
    pairs = sorted(zip(grams_list, ingredients), reverse=True)
    top   = [ing.lower().strip() for _, ing in pairs[:3]]
    if len(top) == 1: return top[0]
    if len(top) == 2: return f"{top[0]} and {top[1]}"
    return f"{top[0]}, {top[1]} and {top[2]}"

def clean_ingredient(s):
    return str(s).lower().strip()

def per_100g(value, total_mass):
    if total_mass <= 0: return 0.0
    return round((float(value) / float(total_mass)) * 100, 1)

def format_example(row):
    ingredients = [clean_ingredient(i) for i in row["ingredients"]]
    grams_list  = [float(g) for g in row["grams_list"]]
    total_mass  = float(row[NUT_MASS])

    if total_mass <= 0 or len(ingredients) < 2:
        return None

    # Remove very short / junk ingredient strings
    ingredients = [i for i in ingredients if len(i) > 2]
    if len(ingredients) < 2:
        return None

    dish_name = make_dish_name(ingredients, grams_list)

    nutrition = {
        "calories":  per_100g(row[NUT_CAL],  total_mass),
        "protein_g": per_100g(row[NUT_PRO],  total_mass),
        "carbs_g":   per_100g(row[NUT_CARB], total_mass),
        "fat_g":     per_100g(row[NUT_FAT],  total_mass),
    }

    

    if not (5 < nutrition["calories"] < 700):  return None
    if nutrition["protein_g"] > 80:            return None
    if nutrition["carbs_g"]   > 150:           return None
    if nutrition["fat_g"]     > 80:            return None
    if any(v < 0 for v in nutrition.values()): return None

    text = (
        "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
        "You are a nutrition assistant for a fitness app. "
        "No explanation. No extra text.<|eot_id|>\n"
        "<|start_header_id|>user<|end_header_id|>\n"
        f'I ate "{dish_name}". List the ingredients and give estimated nutrition per 100g.\n'
        
        "{\n"
        '  "ingredients": ["ingredient1", "ingredient2"],\n'
        '  "nutrition_per_100g": {\n'
        '    "calories": 0,\n'
        '    "protein_g": 0,\n'
        '    "carbs_g": 0,\n'
        '    "fat_g": 0\n'
        "  }\n"
        "}<|eot_id|>\n"
        "<|start_header_id|>assistant<|end_header_id|>\n"
        "{\n"
        f'  "ingredients": {json.dumps(ingredients)},\n'
        f'  "nutrition_per_100g": {json.dumps(nutrition)}\n'
        "}<|eot_id|>"
    )
    return {"text": text}


df = df.sample(n=800, random_state=42)
print("\n Formatting training examples...")
training_data = []
skipped = 0

for _, row in df.iterrows():
    try:
        example = format_example(row)
        if example:
            training_data.append(example)
        else:
            skipped += 1
    except Exception:
        skipped += 1

print(f" {len(training_data)} training examples ready | {skipped} skipped (bad/sparse data)")
print(f"\n Sample:")
print(training_data[0]["text"])

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 512

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '⚠️  No GPU!'}")
print("  Loading Llama 3 8B Instruct (2-4 mins)...\n")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "meta-llama/Meta-Llama-3-8B-Instruct",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,      
    load_in_4bit   = True,      
)

print("loaded")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 8,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = 16,
    lora_dropout   = 0.05,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
    use_rslora     = False,
    loftq_config   = None,
)

model.print_trainable_parameters()


In [ ]:
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import torch

hf_dataset = Dataset.from_list(training_data)
print(f"Training on {len(hf_dataset)} examples")

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = not use_bf16

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = hf_dataset,
    args = SFTConfig(
        output_dir                  = "./food_model_checkpoints",
        num_train_epochs            = 1,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps                = 10,
        learning_rate               = 2e-4,
        fp16                        = use_fp16,
        bf16                        = use_bf16,
        logging_steps               = 25,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "linear",
        seed                        = 42,
        save_total_limit            = 1,
        report_to                   = "none",
        dataset_text_field          = "text",
        max_seq_length              = MAX_SEQ_LENGTH,
        dataset_num_proc            = 1,   
        packing                     = False,
    ),
)

print("Training started")
stats = trainer.train()

mins = stats.metrics["train_runtime"] / 60
loss = stats.metrics["train_loss"]
print(f"\n Training Done")
print(f"     Time : {mins:.1f} minutes")
print(f"    Loss : {loss:.4f}")

In [ ]:
import json, torch
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

def ask_food_ai(dish_name):
    prompt = (
        "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
        "You are a nutrition assistant for a fitness app. "
        "Reply ONLY in valid JSON. No explanation. No extra text.<|eot_id|>\n"
        "<|start_header_id|>user<|end_header_id|>\n"
        f'I ate "{dish_name.lower()}". List the ingredients and give estimated nutrition per 100g.\n'
        "Reply ONLY in this exact JSON format:\n"
        '{\n  "ingredients": ["ingredient1", "ingredient2"],\n'
        '  "nutrition_per_100g": {\n    "calories": 0,\n    "protein_g": 0,\n'
        '    "carbs_g": 0,\n    "fat_g": 0\n  }\n}<|eot_id|>\n'
        "<|start_header_id|>assistant<|end_header_id|>\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = 300,
            temperature        = 0.1,
            do_sample          = True,
            repetition_penalty = 1.1,
            pad_token_id       = tokenizer.eos_token_id,
        )

    raw = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    try:
        s, e = raw.find("{"), raw.rfind("}") + 1
        return json.loads(raw[s:e]) if s != -1 and e > s else {"error": "no JSON", "raw": raw[:100]}
    except json.JSONDecodeError:
        return {"error": "parse failed", "raw": raw[:100]}


TEST_DISHES = [
    "butter chicken",
    "mac and cheese",
    "beef burger",
    "spaghetti bolognese",
    "grilled cheese sandwich",
]

print(" Testing fine-tuned model...\n")
for dish in TEST_DISHES:
    result = ask_food_ai(dish)
    print(f"  {dish.upper()}")
    if "error" in result:
        print(f"    {result['error']}")
    else:
        ings = result.get("ingredients", [])
        nut  = result.get("nutrition_per_100g", {})
        print(f"   Ingredients ({len(ings)}): {', '.join(ings[:4])}{'...' if len(ings)>4 else ''}")
        print(f"   Per 100g → {nut.get('calories',0)} kcal | "
              f"{nut.get('protein_g',0)}g protein | "
              f"{nut.get('carbs_g',0)}g carbs | "
              f"{nut.get('fat_g',0)}g fat")
    print()

In [ ]:
# TESTING FINE-TUNED MODEL ON 20 DISHES 

import json, torch
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

def ask_food_ai(dish_name):
    prompt = (
        "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
        "You are a nutrition assistant for a fitness app. "
        "Reply ONLY in valid JSON. No explanation. No extra text.<|eot_id|>\n"
        "<|start_header_id|>user<|end_header_id|>\n"
        f'I ate "{dish_name.lower()}". List the ingredients and give estimated nutrition per 100g.\n'
        "Reply ONLY in this exact JSON format:\n"
        '{\n  "ingredients": ["ingredient1", "ingredient2"],\n'
        '  "nutrition_per_100g": {\n    "calories": 0,\n    "protein_g": 0,\n'
        '    "carbs_g": 0,\n    "fat_g": 0\n  }\n}<|eot_id|>\n'
        "<|start_header_id|>assistant<|end_header_id|>\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = 300,
            temperature        = 0.1,
            do_sample          = True,
            repetition_penalty = 1.1,
            pad_token_id       = tokenizer.eos_token_id,
        )

    raw = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    try:
        s, e = raw.find("{"), raw.rfind("}") + 1
        return json.loads(raw[s:e]) if s != -1 and e > s else None
    except:
        return None




after = {}
for dish in before.keys():
    result = ask_food_ai(dish)
    if result:
        ings = len(result.get("ingredients", []))
        cal  = result.get("nutrition_per_100g", {}).get("calories", 0)
        after[dish] = {"ingredients": ings, "calories": cal}
    else:
        after[dish] = {"ingredients": 0, "calories": 0}
    print(f"  {dish}")





In [ ]:
import os

SAVE_PATH = "/content/my_food_ai"

print(f"Saving to {SAVE_PATH}...")
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

files_saved = os.listdir(SAVE_PATH)
total_mb = sum(os.path.getsize(f"{SAVE_PATH}/{f}") for f in files_saved) / (1024*1024)
print(f"\n Saved! {len(files_saved)} files | {total_mb:.0f} MB total")

In [ ]:
REPO_NAME  = "my-food-ai"
full_repo  = f"{HF_USERNAME}/{REPO_NAME}"

print(f" Uploading to: https://huggingface.co/{full_repo}")
print("   May take 5-10 mins...\n")

model.push_to_hub(full_repo, private=True, token=HF_TOKEN)
tokenizer.push_to_hub(full_repo, private=True, token=HF_TOKEN)

print(f"\n Done! Your model: https://huggingface.co/{full_repo}")
print(f'\n   To use in Notebook 1:')
print(f'   model_name = "{full_repo}"')